# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [64]:
# imports

import os
import json
import base64
import tempfile
import wave
from dotenv import load_dotenv
from openai import OpenAI
from google import genai
from google.genai import types
import gradio as gr
import sqlite3

In [55]:
# Initialization

# Initialization
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url = "http://localhost:11434/v1"

ollama = OpenAI(api_key="ollama", base_url=ollama_url)
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

# Native Gemini client for TTS (generateContent + AUDIO). OpenAI-compatible /audio/speech is not supported on this host.
gemini_tts = genai.Client(api_key=google_api_key)
TTS_MODEL = "gemini-2.5-flash-preview-tts"
TTS_VOICE_NAME = "Kore"  # prebuilt voice; see https://ai.google.dev/gemini-api/docs/speech-generation

system_message = "You are a helpful assistant"
GMODEL = 'gemini-2.5-flash-lite-tts'
AMODEL = 'claude-haiku-4-5-20251001'
OMODEL = 'x/flux2-klein:latest'

DB = "prices.db"

Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AIzaSyAX


In [56]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [57]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [58]:
get_ticket_price("Paris")

DATABASE TOOL CALLED: Getting price for Paris


'Ticket price to Paris is $899.0'

In [59]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": price_function}]
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [ ]:

def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=GMODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

In [9]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model=GMODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = gemini.chat.completions.create(model=GMODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [8]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

## A bit more about what Gradio actually does:

1. Gradio constructs a frontend Svelte app based on our Python description of the UI
2. Gradio starts a server built upon the Starlette web framework listening on a free port that serves this React app
3. Gradio creates backend routes for our callbacks, like chat(), which calls our functions

And of course when Gradio generates the frontend app, it ensures that the the Submit button calls the right backend route.

That's it!

It's simple, and it has a result that feels magical.

# Let's go multi-modal!!

We can use DALL-E-3, the image generation model behind GPT-4o, to make us some images

Let's put this in a function called artist.

### Price alert: each time I generate an image it costs about 4 cents - don't go crazy with images!

In [11]:
# Some imports for handling images

from io import BytesIO
from PIL import Image

In [34]:
def artist(city):
    print(f"IMAGE GEN CALLED: Generating image for {city}", flush=True)
    image_response = ollama.images.generate(
            model=OMODEL,
            prompt=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
            size="1024x1024",
            n=1,
            response_format="b64_json",
        )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

In [ ]:
image = artist("Tokyo")
display(image)

In [65]:
def _tts_pcm_from_response(response):
    """Decode raw PCM from a generate_content TTS response; handles missing/blocked content."""
    pf = getattr(response, "prompt_feedback", None)
    br = getattr(pf, "block_reason", None) if pf else None
    if br and str(br) not in ("BLOCKED_REASON_UNSPECIFIED", "None"):
        print(f"TTS: prompt blocked: {br}", flush=True)

    for cand in response.candidates or []:
        if cand.content is None:
            print(
                f"TTS: candidate has no content (finish_reason={cand.finish_reason!r}, message={cand.finish_message!r})",
                flush=True,
            )
            continue
        for part in cand.content.parts or []:
            blob = part.inline_data
            if blob is not None and blob.data:
                raw = blob.data
                return base64.b64decode(raw) if isinstance(raw, str) else raw
    return None


def _tts_config():
    """TTS-oriented safety: assistant price lines are often flagged if thresholds are too strict."""
    return types.GenerateContentConfig(
        response_modalities=["AUDIO"],
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(
                    voice_name=TTS_VOICE_NAME,
                )
            )
        ),
        safety_settings=[
            types.SafetySetting(
                category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
                threshold=types.HarmBlockThreshold.BLOCK_ONLY_HIGH,
            ),
            types.SafetySetting(
                category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
                threshold=types.HarmBlockThreshold.BLOCK_ONLY_HIGH,
            ),
            types.SafetySetting(
                category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
                threshold=types.HarmBlockThreshold.BLOCK_ONLY_HIGH,
            ),
            types.SafetySetting(
                category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
                threshold=types.HarmBlockThreshold.BLOCK_ONLY_HIGH,
            ),
        ],
    )


def talker(message):
    print(f"TTS TOOL CALLED: Generating audio for {message}", flush=True)
    text = (message or "").strip()
    if not text:
        return None

    snippet = text[:5000]
    cfg = _tts_config()

    response = gemini_tts.models.generate_content(
        model=TTS_MODEL,
        contents=snippet,
        config=cfg,
    )
    pcm = _tts_pcm_from_response(response)
    if pcm is None:
        response = gemini_tts.models.generate_content(
            model=TTS_MODEL,
            contents=f"Say clearly: {snippet}",
            config=cfg,
        )
        pcm = _tts_pcm_from_response(response)

    if pcm is None:
        print("TTS: no audio bytes in response (after retry)", flush=True)
        return None

    tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
    path = tmp.name
    tmp.close()
    with wave.open(path, "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(24000)
        wf.writeframes(pcm)

    print(f"TTS TOOL CALLED: Generated audio for {message}", flush=True)
    return path

## Let's bring this home:

1. A multi-modal AI assistant with image and audio generation
2. Tool callling with database lookup
3. A step towards an Agentic workflow


In [61]:
def chat(history):
    print(f"CHAT CALLED: Generating response for {history}", flush=True)
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = gemini.chat.completions.create(model="gemini-3.1-flash-lite-preview", messages=messages, tools=tools)
    cities = []
    image = None

    while response.choices[0].finish_reason=="tool_calls":
        print(f"CHAT TOOL CALLED: Generating response for {history}", flush=True)
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = gemini.chat.completions.create(model="gemini-3.1-flash-lite-preview", messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    voice = talker(reply)

    if cities:
        image = artist(cities[0])
    
    return history, voice, image


In [62]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            cities.append(city)
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses, cities

## The 3 types of Gradio UI

`gr.Interface` is for standard, simple UIs

`gr.ChatInterface` is for standard ChatBot UIs

`gr.Blocks` is for custom UIs where you control the components and the callbacks

In [66]:
# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7886
* To create a public link, set `share=True` in `launch()`.


CHAT CALLED: Generating response for [{'role': 'user', 'metadata': None, 'content': 'Hey there!', 'options': None}]
TTS TOOL CALLED: Generating audio for Hello! How can I assist you with your travel plans today?
TTS TOOL CALLED: Generated audio for Hello! How can I assist you with your travel plans today?
CHAT CALLED: Generating response for [{'role': 'user', 'metadata': None, 'content': 'Hey there!', 'options': None}, {'role': 'assistant', 'metadata': None, 'content': 'Hello! How can I assist you with your travel plans today?', 'options': None}, {'role': 'user', 'metadata': None, 'content': "I'd like to go to London", 'options': None}]
CHAT TOOL CALLED: Generating response for [{'role': 'user', 'content': 'Hey there!'}, {'role': 'assistant', 'content': 'Hello! How can I assist you with your travel plans today?'}, {'role': 'user', 'content': "I'd like to go to London"}]
DATABASE TOOL CALLED: Getting price for London
TTS TOOL CALLED: Generating audio for The price for a return ticket to

# Exercises and Business Applications

Add in more tools - perhaps to simulate actually booking a flight. A student has done this and provided their example in the community contributions folder.

Next: take this and apply it to your business. Make a multi-modal AI assistant with tools that could carry out an activity for your work. A customer support assistant? New employee onboarding assistant? So many possibilities! Also, see the week2 end of week Exercise in the separate Notebook.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a HUGE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>